# M4 — Escalation risk

**Approach:** escalation risk model (contributor E). Class-balanced logistic regression on summary + tags + category + channel + month + tenure + tag-count. **Chronological 70/15/15 split**; the decision threshold is tuned on the validation set to a **minimum-recall = 0.80** target (don't miss escalations), then ROC-AUC, PR-AUC, and precision/recall at that threshold are reported on the test set. Saves `models/m4_escalation.joblib`.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.exists(os.path.join(_p, "northstar_prep.py")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import numpy as np, pandas as pd
import northstar_prep as P

In [ ]:
df = P.load_clean()
df["escalated"] = df["escalated"].astype(bool)
FEATS, TARGET, POS, MIN_RECALL = ["case_summary","tags","category","channel","month_num",
                                  "customer_tenure_months","n_tags"], "escalated", True, 0.80
labels = [False, True]
tr, va, te = P.time_split(df)
trans = [P.t_word(), P.t_char(), P.t_tags(), P.t_cat(["category","channel"]),
         P.t_num(["month_num","customer_tenure_months","n_tags"])]
pipe = P.make_clf(trans).fit(tr[FEATS], tr[TARGET])
pos_i = list(pipe.classes_).index(POS)
pv = pipe.predict_proba(va[FEATS])[:, pos_i]
thr = P.recall_targeted_threshold((va[TARGET] == POS), pv, min_recall=MIN_RECALL)

In [ ]:
pt = pipe.predict_proba(te[FEATS])[:, pos_i]
pred = np.where(pt >= thr, True, False)
base = (~df[TARGET]).mean()
metrics = P.evaluate(te[TARGET], pred, labels, proba=pt, positive=POS, baseline=base)
metrics.update(P.binary_at_threshold(te[TARGET], pt, POS, thr))
metrics["min_recall_target"] = MIN_RECALL
metrics

In [ ]:
test = te[["case_id","case_summary",TARGET]].copy().reset_index(drop=True)
test["prob"] = pt.round(3); test["predicted"] = pred; test["confidence"] = test["prob"]
full = P.make_clf(trans).fit(df[FEATS], df[TARGET])
P.save_bundle("m4_escalation", {
    "model": full, "target": TARGET, "features": FEATS, "classes": labels, "kind": "binary",
    "label": "Escalation risk", "threshold": round(float(thr), 3), "positive": POS,
    "metrics": metrics, "test": test,
    "report": P.class_report(te[TARGET], pred, labels), "confusion": P.conf_matrix(te[TARGET], pred, labels),
    "meta": {"contributor":"E", "approach":"escalation risk, recall-targeted threshold",
             "split":"time 70/15/15", "threshold_rule":"min recall 0.80 on validation",
             "inputs":"summary + tags + category + channel + month + tenure + n_tags"}})